# GCP2 Notes: SCE-UA calibration workflow for GR4J

## Notebook purpose
This notebook calibrates the GR4J rainfall-runoff model across multiple catchments using the **SCE-UA** optimization framework (with the option to switch to Latin Hypercube Sampling). The code loops through available basin files, runs the calibration routine, extracts the best parameter set, and writes a summary table of results.

## Why this notebook matters in the project
The later regionalization notebooks depend on calibrated GR4J parameters such as `x1`, `x2`, `x3`, and `x4`. This notebook is the upstream step that generates those parameter estimates and associated calibration/validation skill values. In other words, this notebook produces the parameter dataset that the machine learning and KNN notebooks later use as input.

## High-level workflow
The code below performs four main tasks:
1. define calibration settings and file paths,
2. define helper functions for evaluation and model execution,
3. loop through all catchment data files,
4. save the best parameter estimates and skill metrics to a CSV file.

The markdown added here explains each of those stages so the notebook reads as a documented workflow rather than a single large block of code.


## Main calibration block

### Purpose of this block
This code cell contains the complete calibration pipeline for GR4J. It sets the experiment options, defines helper functions, runs the chosen search algorithm basin by basin, and stores the best results.

### Detailed walkthrough

#### 1. Imports and external dependencies
The notebook begins by importing:
- core scientific Python packages such as `pandas` and `numpy`,
- the `spotpy` library for parameter search,
- the `hydrogr` package
- date/time tools for defining calibration and validation windows,
- file-path utilities, and
- the hydrologic model classes needed to load forcing data and run GR4J.

This makes it clear that the workflow combines general data handling with a domain-specific model implementation.

#### 2. Settings section
The settings block defines the full experiment configuration, including:
- the directory containing catchment input files,
- the optimization algorithm (`sceua` or `lhs`),
- the number of repetitions,
- the random seed,
- the warmup length,
- the calibration period,
- the validation period, and
- the output CSV path.

This section is important because it contains the assumptions that govern the entire calibration exercise. It is the first place to edit if the study period, algorithm, or storage location needs to change.

#### 3. Objective function: NSE
A helper function computes the Nash-Sutcliffe Efficiency (NSE), which is used here as the core performance criterion. The function:
- converts inputs to numeric arrays,
- removes non-finite values,
- checks for empty or degenerate cases, and
- returns a safe value when NSE cannot be computed.

Encapsulating NSE in its own function improves readability and keeps the later calibration logic focused on workflow rather than formula details.

#### 4. GR4J run helper
A second helper function runs the GR4J model for a given basin and parameter set. Conceptually, this function is responsible for connecting the input data to the hydrologic model, executing the simulation, and returning outputs that can be evaluated against observations.

This separation is good practice because model execution is a distinct task from result summarization.

#### 5. Best-parameter extraction helper
Another helper function isolates the best solution from the optimization output. This is useful because calibration algorithms often generate many candidate parameter sets, but the later regionalization workflow needs one clear “best” parameter set per catchment.

#### 6. Main loop over basins
The largest part of the notebook iterates over the available catchment files. For each basin, the code:
- loads the basin data,
- prepares the calibration problem,
- selects the requested search algorithm,
- runs the optimizer,
- extracts the best parameter set,
- evaluates calibration and validation performance, and
- appends the results to a growing list.

This loop is the operational core of the notebook because it scales the calibration procedure from one basin to the full study set.

#### 7. Save results
At the end, the collected results are written to a CSV file. This output becomes a portable dataset that can be used in downstream notebooks without recalibrating the model every time.

### Why this block matters
Although this notebook appears short, it performs a foundational role in the project. The quality of the later regionalization analysis depends directly on the quality and consistency of the parameter estimates generated here.


In [1]:
import pandas as pd
import numpy as np
import spotpy
import datetime as dt
from pathlib import Path
import gc
from hydrogr import InputDataHandler, ModelGr4j

# === 1. Settings ===
DATA_DIR = Path.home() / "Downloads" / "CAMELS_US_GR4J"

ALGO = "sceua"          # "lhs" or "sceua"
N_REPS = 10000
SEED = 42
WARMUP = 365

CAL_PERIOD = (dt.datetime(1980, 1, 1), dt.datetime(2004, 12, 31))
VAL_PERIOD = (dt.datetime(2005, 1, 1), dt.datetime(2014, 12, 31))

OUT_PATH = Path.home() / "Downloads" / f"gr4j_{ALGO}_{N_REPS}.csv"

np.random.seed(SEED)


# === 2. Core Functions ===
def nse(sim, obs):
    sim = np.asarray(sim, dtype=float)
    obs = np.asarray(obs, dtype=float)

    mask = np.isfinite(sim) & np.isfinite(obs)
    sim = sim[mask]
    obs = obs[mask]

    if len(obs) == 0:
        return np.nan

    denom = np.sum((obs - np.mean(obs)) ** 2)
    if denom == 0:
        return np.nan

    return 1 - np.sum((obs - sim) ** 2) / denom


class SpotpySetup:
    def __init__(self, data, algo="lhs", warmup=365):
        self.data = data
        self.algo = algo
        self.warmup = warmup
        self.inputs = InputDataHandler(ModelGr4j, self.data)

        self.params = [
            spotpy.parameter.Uniform("x1", 0.01, 3000.0),
            spotpy.parameter.Uniform("x2", -5.0, 5.0),
            spotpy.parameter.Uniform("x3", 0.01, 300.0),
            spotpy.parameter.Uniform("x4", 0.5, 10.0),
        ]

    def parameters(self):
        return spotpy.parameter.generate(self.params)

    def evaluation(self):
        return self.data["flow_mm"].values[self.warmup:]

    def simulation(self, vector):
        model = ModelGr4j({
            "X1": vector[0],
            "X2": vector[1],
            "X3": vector[2],
            "X4": vector[3],
        })
        sim = model.run(self.inputs.data)["flow"].values
        return sim[self.warmup:]

    def objectivefunction(self, simulation, evaluation):
        score = nse(simulation, evaluation)

        if np.isnan(score):
            return 1e6 if self.algo == "sceua" else -1e6

        if self.algo == "sceua":
            return 1 - score

        return score


def run_gr4j(df, p):
    model = ModelGr4j({
        "X1": p["x1"],
        "X2": p["x2"],
        "X3": p["x3"],
        "X4": p["x4"],
    })
    return model.run(df)["flow"].values


def extract_best_parameters(results, algo):
    if algo == "sceua":
        bestindex, _ = spotpy.analyser.get_minlikeindex(results)
        best = results[bestindex]
    else:
        bestindex = np.argmax(results["like1"])
        best = results[bestindex]

    params = {}
    for name in best.dtype.names:
        if name.startswith("par"):
            clean_name = name.replace("par", "")
            params[clean_name] = float(best[name])

    return params, best


# === 3. Main Loop ===
csv_files = sorted(DATA_DIR.glob("*.csv"))
results = []

algos = {
    "lhs": spotpy.algorithms.lhs,
    "sceua": spotpy.algorithms.sceua,
}

if ALGO not in algos:
    raise ValueError(f"Unsupported ALGO: {ALGO}. Use 'lhs' or 'sceua'.")

if not DATA_DIR.exists():
    raise FileNotFoundError(f"Dataset folder not found: {DATA_DIR}")

if len(csv_files) == 0:
    raise FileNotFoundError(f"No CSV files found in: {DATA_DIR}")

sampler_class = algos[ALGO]

for csv_path in csv_files:
    try:
        # Load data
        df = pd.read_csv(csv_path, parse_dates=["Date"], index_col="Date")
        df.columns = ["precipitation", "evapotranspiration", "flow_mm"]

        # Split periods
        df_cal = df[CAL_PERIOD[0]:CAL_PERIOD[1]].copy()
        df_val = df[VAL_PERIOD[0]:VAL_PERIOD[1]].copy()

        # Skip files with too little data
        if len(df_cal) <= WARMUP or len(df_val) <= WARMUP:
            print(f"Skipped {csv_path.stem}: not enough data after warmup")
            continue

        # Setup sampler
        setup = SpotpySetup(df_cal, algo=ALGO, warmup=WARMUP)
        sampler = sampler_class(setup, dbformat="ram")

        # Run calibration
        sampler.sample(N_REPS)

        spotpy_results = sampler.getdata()
        params, best_row = extract_best_parameters(spotpy_results, ALGO)

        # Full simulations for reporting
        q_cal_sim = run_gr4j(df_cal, params)[WARMUP:]
        q_val_sim = run_gr4j(df_val, params)[WARMUP:]

        obs_cal = df_cal["flow_mm"].values[WARMUP:]
        obs_val = df_val["flow_mm"].values[WARMUP:]

        # Store results
        res = {
            "catchment": csv_path.stem,
            **params,
            "NSE_cal": nse(q_cal_sim, obs_cal),
            "NSE_val": nse(q_val_sim, obs_val),
            "objective_value": float(best_row["like1"]),
        }

        results.append(res)

        print(
            f"Done: {csv_path.stem} | "
            f"Cal NSE: {res['NSE_cal']:.3f} | "
            f"Val NSE: {res['NSE_val']:.3f}"
        )

        del setup
        del sampler
        del spotpy_results
        del df
        del df_cal
        del df_val
        gc.collect()

    except Exception as e:
        print(f"Failed {csv_path.stem}: {e}")

# Save results
pd.DataFrame(results).to_csv(OUT_PATH, index=False)
print(f"\nSaved results to: {OUT_PATH}")
print(f"Processed catchments: {len(results)}")

Initializing the  Shuffled Complex Evolution (SCE-UA) algorithm  with  10000  repetitions
The objective function will be minimized
Starting burn-in sampling...
Initialize database...
['csv', 'hdf5', 'ram', 'sql', 'custom', 'noData']
Burn-in sampling completed...
Starting Complex Evolution...
ComplexEvo loop #1 in progress...
ComplexEvo loop #2 in progress...
ComplexEvo loop #3 in progress...
ComplexEvo loop #4 in progress...
ComplexEvo loop #5 in progress...
2394 of 10000, minimal objective function=0.922152, time remaining: 00:00:06
ComplexEvo loop #6 in progress...
ComplexEvo loop #7 in progress...
ComplexEvo loop #8 in progress...
ComplexEvo loop #9 in progress...
ComplexEvo loop #10 in progress...
ComplexEvo loop #11 in progress...
5034 of 10000, minimal objective function=0.921937, time remaining: 00:00:04
ComplexEvo loop #12 in progress...
ComplexEvo loop #13 in progress...
ComplexEvo loop #14 in progress...
ComplexEvo loop #15 in progress...
ComplexEvo loop #16 in progress...
76

## Interpretation and quality checks

After running this notebook, the main checks to make are:
- whether all expected basins were processed successfully,
- whether calibration and validation NSE values are reasonable,
- whether any basins failed or returned suspicious parameter values, and
- whether the saved CSV contains the fields needed by the downstream regionalization notebooks.

This is an important checkpoint because poor or inconsistent calibration outputs would propagate into every later analysis step.
